<a href="https://colab.research.google.com/github/bada9724-afk/stock-app/blob/main/2026___ai__2__.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎖️ 2026년 부대 맞춤형 AI·SW 소양교육
## 육군 직무보수교육 - **2일차 실습 노트북**

> **Ⅱ. 데이터 수집 및 전처리**
>  
> - 데이터 수집 개념 및 공공데이터 활용 (data.go.kr)
> - pandas를 이용한 CSV / Excel / JSON 데이터 읽기
> - DataFrame 탐색 (head, info, describe)
> - 데이터 선택·필터링 (loc, iloc, Boolean Indexing)
> - 결측값(Missing Value) 처리
> - 중복 제거·타입 변환·정규화
> - 이상치(Outlier) 탐지 및 처리
> - 데이터 그룹화 & 집계 (groupby)

---

### 📌 노트북 사용법
1. 각 셀을 순서대로 실행합니다. (`Shift + Enter` 또는 ▶ 버튼)
2. 💻 **실습 예제** 셀은 직접 실행하며 결과를 확인합니다.
3. 🔥 **실습문제** 셀에 직접 답안 코드를 작성해 보세요.
4. ✅ **정답 코드** 셀은 스스로 풀어본 뒤 확인하세요.

### 💡 핵심 원리: Garbage In, Garbage Out
> **"쓰레기가 들어오면, 쓰레기가 나온다"**  
> 데이터 분석의 **80%** 는 수집·전처리에 소요됩니다.  
> 수집 단계가 **분석 품질을 결정**합니다.

---

# 🛠️ Chapter 0. 환경 구성 및 샘플 데이터 준비

## 0-1. 주요 라이브러리 import
pandas, numpy, matplotlib은 Colab에 기본 설치되어 있습니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# 버전 확인
print(f"pandas  : {pd.__version__}")
print(f"numpy   : {np.__version__}")
print(f"mpl     : {mpl.__version__}")

## 0-2. 한글 폰트 설정 (Colab 그래프 한글 깨짐 방지)

In [ ]:
# 나눔고딕 폰트 설치
!apt-get install -qq fonts-nanum > /dev/null 2>&1

import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')

plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)

# 테스트
plt.figure(figsize=(5, 2))
plt.title("한글 폰트 테스트 — 육군 AI 소양교육")
plt.plot([1, 2, 3], [10, 20, 15])
plt.xlabel("순서")
plt.ylabel("값")
plt.show()
print("✅ 한글 폰트 설정 완료")

## 0-3. 실습용 샘플 데이터 생성

외부 파일이 없어도 실습할 수 있도록, **부대원 인사/훈련 데이터** 를 Colab 환경에 직접 만듭니다.

- 의도적으로 **결측값(NaN), 중복 행, 이상치** 를 일부 포함시켰습니다.

In [ ]:
import pandas as pd
import numpy as np

# 재현 가능한 난수
np.random.seed(42)

# 부대원 샘플 데이터
data = {
    "이름":     ["김철수", "이영희", "박민준", "최수진", "정태양",
               "한소희", "윤도현", "강민아", "송지훈", "임서연",
               "오현우", "배유진", "신동욱", "조은비", "문재혁",
               "한소희", "이수민", "박지성", "권나라", "최우식"],
    "부서":     ["행정과", "보급과", "통신과", "행정과", "보급과",
               "통신과", "행정과", "보급과", "통신과", "행정과",
               "보급과", "통신과", "행정과", "보급과", "통신과",
               "통신과", "행정과", "보급과", "행정과", "통신과"],
    "계급":     ["상병", "병장", "이병", "일병", "상병",
               "병장", "이병", "일병", "상병", "병장",
               "이병", "일병", "상병", "병장", "이병",
               "병장", "상병", "일병", "이병", "병장"],
    "입사연도": [2019, 2021, 2020, 2018, 2019,
               2022, 2023, 2021, 2020, 2018,
               2023, 2022, 2019, 2018, 2023,
               2022, 2020, 2021, 2023, 2018],
    "급여":    [420, 380, np.nan, 450, 410,
               390, 250, 350, 430, 470,
               260, 340, 420, 480, 255,
               390, 415, 355, np.nan, 475],
    "훈련점수": [85.2, 92.3, 78.5, 95.1, 88.0,
                np.nan, 72.1, 80.5, 91.2, 87.8,
                75.0, 82.4, 89.6, 94.3, 9999,    # 9999는 이상치
                92.3, 86.7, 79.2, 70.5, 93.1],
    "출석률":   [95.2, 88.7, 72.1, 91.5, 94.8,
                83.2, 78.9, 85.4, 90.1, 96.3,
                75.5, 81.7, 89.4, 97.2, 68.5,
                88.7, 92.3, 84.6, np.nan, 95.7],
    "입대일":   ["2019-03-01", "2021-06-15", "2020-09-10", "2018-04-20", "2019-07-05",
               "2022-02-28", "2023-05-12", "2021-11-03", "2020-08-17", "2018-10-25",
               "2023-01-09", "2022-07-14", "2019-12-22", "2018-05-30", "2023-03-18",
               "2022-02-28", "2020-04-05", "2021-09-19", "2023-06-24", "2018-11-08"]
}

df_original = pd.DataFrame(data)

# CSV로 저장하여 이후 실습에서 파일 읽기 연습에도 사용
df_original.to_csv('/content/soldiers.csv', index=False, encoding='utf-8-sig')
df_original.to_excel('/content/soldiers.xlsx', index=False)

print(f"✅ 샘플 데이터 생성 완료: {df_original.shape[0]}행 × {df_original.shape[1]}열")
print(f"   저장 위치: /content/soldiers.csv, /content/soldiers.xlsx")
df_original.head()

---
# 📥 Chapter 1. 데이터 수집 (Data Collection)

## 1-1. 데이터 수집이란?
> **분석 목적에 맞는 데이터를 찾고 불러오는 과정**  
> 데이터 분석의 첫 단계이자 가장 중요한 단계입니다.

### 5가지 주요 데이터 수집 방법

| 방법 | 도구 | 특징 |
|---|---|---|
| 🏛 **공공 데이터포털** | data.go.kr | 정부·지자체 무료 제공 |
| 📁 **파일 직접 읽기** | pandas | CSV, Excel, JSON 로컬 파일 |
| 🌐 **웹 스크래핑** | requests + BeautifulSoup | HTML 파싱 |
| 🔌 **API 호출** | requests | REST API, JSON 수신 |
| 🗄 **데이터베이스** | pandas.read_sql() | SQL 쿼리 |

## 1-2. 공공데이터포털 활용 절차
1. **data.go.kr** 회원가입·로그인
2. 원하는 데이터 검색 (예: 날씨, 인구, 교통)
3. 오픈API → 활용신청 → 즉시 승인
4. 마이페이지에서 **API 인증키(서비스키)** 확인

💡 군 관련 활용 가능 공공 데이터 예시:
- 기상청 날씨 예보
- 행정안전부 지역 정보
- 통계청 인구 데이터
- 보건복지부 의료기관
- 교통 CCTV 정보

## 1-3. API 호출 코드 구조 (참고)

아래는 **일반적인 공공 API 호출 패턴**입니다. 실제 실행하려면 본인의 인증키가 필요하므로, 코드 구조만 익혀두세요.

In [ ]:
# ⚠️ 이 셀은 '구조 학습용'입니다. 실제 API 키가 없으면 실행되지 않습니다.

import requests
import pandas as pd

# 공공데이터 API 설정 (예시)
# KEY = '본인_인증키'
# URL = 'http://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getVilageFcst'
#
# params = {
#     'serviceKey': KEY,
#     'numOfRows': 100,
#     'pageNo': 1,
#     'dataType': 'JSON',
#     'base_date': '20260428',
#     'base_time': '0600',
#     'nx': 60, 'ny': 127   # 좌표
# }
#
# r = requests.get(URL, params=params)
# data = r.json()
# df = pd.DataFrame(data['response']['body']['items']['item'])
# df.head()

print("👆 실제 실행하려면 주석 해제 후 본인의 API 인증키를 입력하세요.")
print("   data.go.kr 회원가입 → 데이터 검색 → 활용신청 → 인증키 발급")

---
### 🔥 실습문제 1
아래 질문에 답해보세요. (`answer` 변수에 숫자로 답 저장)

1. 데이터 분석에서 수집·전처리 단계가 차지하는 비율은 몇 %인가?
2. 공공데이터포털의 정식 URL은?  
   (① kosis.kr ② data.go.kr ③ open.go.kr)

In [ ]:
# ✍️ 여기에 답을 작성하세요
answer_1 = None     # 숫자 (%) 입력
answer_2 = None     # 1, 2, 3 중 하나

print(f"답 1: {answer_1}%")
print(f"답 2: {answer_2}번")

**✅ 정답**

In [ ]:
answer_1 = 80         # 80% (PPT 내용)
answer_2 = 2          # ② data.go.kr

print(f"답 1: {answer_1}% → 데이터 분석의 80%는 수집·전처리에 소요")
print(f"답 2: {answer_2}번 → data.go.kr (공공데이터포털)")

---
# 📂 Chapter 2. pandas로 다양한 형식의 데이터 읽기

> **pandas** = 데이터를 **DataFrame(표 형태)** 으로 읽어 분석하기 쉽게 만들어주는 Python 핵심 라이브러리

## 2-1. CSV 파일 읽기 — `pd.read_csv()`

In [1]:
import pandas as pd

# 기본 읽기
df = pd.read_csv('/content/soldiers.csv')
print(f"Shape: {df.shape}")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/soldiers.csv'

In [ ]:
# 자주 쓰는 옵션
df = pd.read_csv(
    '/content/soldiers.csv',
    encoding='utf-8-sig',    # 한글 인코딩 (cp949, euc-kr 등 가능)
    sep=',',                 # 구분자 (탭은 '\t')
    header=0,                # 첫 행을 헤더로
)
df.head(3)

## 2-2. Excel 파일 읽기 — `pd.read_excel()`

In [ ]:
# 기본 읽기
df_xl = pd.read_excel('/content/soldiers.xlsx')
df_xl.head(3)

In [ ]:
# 주요 옵션 정리
# ─────────────────────────────────────────────────
# sheet_name=0           첫 번째 시트 (기본값)
# sheet_name='3월'       시트 이름으로 선택
# sheet_name=None        모든 시트를 딕셔너리로 반환
# header=1               2번째 행을 헤더로 사용
# skiprows=2             상단 2행 건너뛰기
# usecols='A:D'          A~D 열만 읽기
# engine='openpyxl'      한글 엑셀 파일 깨짐 방지

# 예시: 일부 컬럼만 읽기
df_part = pd.read_excel(
    '/content/soldiers.xlsx',
    usecols='A:D',    # A~D 열만
    engine='openpyxl'
)
df_part.head()

## 2-3. JSON 파일 읽기 — `pd.read_json()`, `json_normalize()`

In [ ]:
import json

# 샘플 JSON 데이터 생성
sample_json = {
    "items": [
        {"이름": "김철수", "계급": "상병", "점수": 85},
        {"이름": "이영희", "계급": "병장", "점수": 92},
        {"이름": "박민준", "계급": "이병", "점수": 78}
    ]
}

# 파일로 저장
with open('/content/soldiers.json', 'w', encoding='utf-8') as f:
    json.dump(sample_json, f, ensure_ascii=False, indent=2)

# 방법 1: pd.DataFrame()
df_json = pd.DataFrame(sample_json['items'])
print("[ 방법 1: DataFrame() ]")
print(df_json)

# 방법 2: json_normalize (중첩 구조도 평탄화)
from pandas import json_normalize
df_norm = json_normalize(sample_json['items'])
print("\n[ 방법 2: json_normalize ]")
print(df_norm)

In [2]:
import json

# 샘플 JSON 데이터 생성
sample_json = {
    "items": [
        {"이름": "김철수", "계급": "상병", "점수": 85},
        {"이름": "이영희", "계급": "병장", "점수": 92},
        {"이름": "박민준", "계급": "이병", "점수": 78}
    ]
}

# 파일로 저장
with open('/content/soldiers.json', 'w', encoding='utf-8') as f:
    json.dump(sample_json, f, ensure_ascii=False, indent=2)

# 방법 1: pd.DataFrame()
df_json = pd.DataFrame(sample_json['items'])
print("[ 방법 1: DataFrame() ]")
print(df_json)

# 방법 2: json_normalize (중첩 구조도 평탄화)
from pandas import json_normalize
df_norm = json_normalize(sample_json['items'])
print("\n[ 방법 2: json_normalize ]")
print(df_norm)

[ 방법 1: DataFrame() ]
    이름  계급  점수
0  김철수  상병  85
1  이영희  병장  92
2  박민준  이병  78

[ 방법 2: json_normalize ]
    이름  계급  점수
0  김철수  상병  85
1  이영희  병장  92
2  박민준  이병  78


## 2-4. Google Drive에서 파일 읽기 (참고)

Drive에 파일이 있는 경우 아래와 같이 마운트 후 경로로 불러옵니다.

```python
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/데이터/military_data.csv')
```

---
### 🔥 실습문제 2
위에서 저장한 `/content/soldiers.csv` 파일을 다음 조건으로 읽어 오세요.

1. 인코딩: `utf-8-sig`
2. 컬럼 `"이름"`, `"계급"`, `"훈련점수"` **3개 열만** 읽기 (`usecols` 옵션)
3. 상위 5행 출력

In [5]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv('/content/drive/MyDrive/military_meal_2026_H1.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**✅ 정답 예시**

In [ ]:
df_selected = pd.read_csv(
    '/content/soldiers.csv',
    encoding='utf-8-sig',
    usecols=['이름', '계급', '훈련점수']
)

print(f"Shape: {df_selected.shape}")
df_selected.head()

---
# 🔍 Chapter 3. 데이터 탐색 — DataFrame 기본 함수

> 💡 데이터를 불러온 후 **가장 먼저** 해야 할 일:  
> 형태·구조·통계를 파악하여 **전처리 전략을 세우기**

## 3-1. 데이터를 처음 받았을 때 확인하는 6가지

| 함수 | 역할 |
|---|---|
| `df.head()` | 상위 5행 미리보기 |
| `df.tail()` | 하위 5행 미리보기 |
| `df.shape` | (행 수, 열 수) 확인 |
| `df.info()` | 컬럼명·자료형·결측값 요약 |
| `df.describe()` | 수치형 통계 요약 (평균, 표준편차, 사분위수) |
| `df.columns` / `df.dtypes` | 컬럼명 / 자료형 목록 |

In [9]:
df = pd.read_csv('/content/soldiers.csv')

# ① 상위 5행
print("─" * 40)
print("① df.head()")
print("─" * 40)
print(df.head())

────────────────────────────────────────
① df.head()
────────────────────────────────────────
    이름   부서  계급  입사연도     급여  훈련점수   출석률         입대일
0  김철수  행정과  상병  2019  420.0  85.2  95.2  2019-03-01
1  이영희  보급과  병장  2021  380.0  92.3  88.7  2021-06-15
2  박민준  통신과  이병  2020    NaN  78.5  72.1  2020-09-10
3  최수진  행정과  일병  2018  450.0  95.1  91.5  2018-04-20
4  정태양  보급과  상병  2019  410.0  88.0  94.8  2019-07-05


In [17]:
# ② 하위 3행
print("② df.tail(3)")
print(df.tail(3))

② df.tail(3)
     이름   부서  계급  입사연도     급여  훈련점수   출석률         입대일
17  박지성  보급과  일병  2021  355.0  79.2  84.6  2021-09-19
18  권나라  행정과  이병  2023    NaN  70.5   NaN  2023-06-24
19  최우식  통신과  병장  2018  475.0  93.1  95.7  2018-11-08


In [18]:
```python
# 📌 라이브러리 설치 (한 번만 실행)
!pip install requests beautifulsoup4 lxml

import requests
from bs4 import BeautifulSoup
from datetime import datetime

# 📌 삼성전자 코드
stock_code = "005930"

# 📌 네이버 금융 URL
url = f"https://finance.naver.com/item/main.nhn?code={stock_code}"

headers = {"User-Agent": "Mozilla/5.0"}

# 📌 데이터 가져오기
res = requests.get(url, headers=headers)
soup = BeautifulSoup(res.text, "lxml")

# 📌 현재 주가
price = soup.select_one(".no_today .blind").text
price = int(price.replace(",", ""))

# 📌 PER / PBR
table = soup.select("table.per_table td")

def to_float(x):
    try:
        return float(x)
    except:
        return None

per = to_float(table[10].text.strip())
pbr = to_float(table[13].text.strip())

# 📌 현재 시간
now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# 📊 투자 판단
if per is not None:
    if per < 8:
        decision = "매수"
    elif per < 12:
        decision = "관망"
    else:
        decision = "매도 고려"
else:
    decision = "판단 불가"

# 📊 목표가 / 손절가
target1 = int(price * 1.15)
target2 = int(price * 1.30)
stop_loss = int(price * 0.90)

# 📊 출력
print("="*40)
print(f"현재시간: {now}")
print(f"현재주가: {price:,}원")
print(f"PER: {per}")
print(f"PBR: {pbr}")
print("-"*40)
print(f"판단: {decision}")
print("-"*40)
print(f"목표가1: {target1:,}원")
print(f"목표가2: {target2:,}원")
print(f"손절가: {stop_loss:,}원")
print("="*40)
```


SyntaxError: invalid syntax (1008694883.py, line 1)

In [19]:
import requests
from bs4 import BeautifulSoup
from datetime import datetime

# 📌 삼성전자 코드

stock_code = "005930"

# 📌 네이버 금융 URL

url = f"https://finance.naver.com/item/main.nhn?code={stock_code}"

headers = {"User-Agent": "Mozilla/5.0"}

# 📌 데이터 가져오기

res = requests.get(url, headers=headers)
soup = BeautifulSoup(res.text, "lxml")

# 📌 현재 주가

price = soup.select_one(".no_today .blind").text
price = int(price.replace(",", ""))

# 📌 PER / PBR

table = soup.select("table.per_table td")

def to_float(x):
try:
return float(x)
except:
return None

per = to_float(table[10].text.strip())
pbr = to_float(table[13].text.strip())

# 📌 현재 시간

now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# 📊 투자 판단

if per is not None:
if per < 8:
decision = "매수"
elif per < 12:
decision = "관망"
else:
decision = "매도 고려"
else:
decision = "판단 불가"

# 📊 목표가 / 손절가

target1 = int(price * 1.15)
target2 = int(price * 1.30)
stop_loss = int(price * 0.90)

# 📊 출력

print("="*40)
print(f"현재시간: {now}")
print(f"현재주가: {price:,}원")
print(f"PER: {per}")
print(f"PBR: {pbr}")
print("-"*40)
print(f"판단: {decision}")
print("-"*40)
print(f"목표가1: {target1:,}원")
print(f"목표가2: {target2:,}원")
print(f"손절가: {stop_loss:,}원")
print("="*40)


IndentationError: expected an indented block after function definition on line 29 (3138039196.py, line 30)

In [20]:
# ③ 데이터 크기
print(f"③ df.shape = {df.shape}")
print(f"   → {df.shape[0]}행 × {df.shape[1]}열")

③ df.shape = (20, 8)
   → 20행 × 8열


In [15]:
# ④ df.info() - 구조 전체 파악
print("④ df.info()")
df.info()

④ df.info()
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   이름      20 non-null     object 
 1   부서      20 non-null     object 
 2   계급      20 non-null     object 
 3   입사연도    20 non-null     int64  
 4   급여      18 non-null     float64
 5   훈련점수    19 non-null     float64
 6   출석률     19 non-null     float64
 7   입대일     20 non-null     object 
dtypes: float64(3), int64(1), object(4)
memory usage: 1.4+ KB


In [14]:
# ⑤ df.describe() - 수치형 통계 요약
print("⑤ df.describe()")
df.describe()

⑤ df.describe()


,입사연도,급여,훈련점수,출석률
count,20.000000,18.000000,19.000000,19.000000
mean,2020.500000,385.555556,606.989474,86.831579
std,1.849609,72.576306,2274.391597,8.420283
min,2018.000000,250.000000,70.500000,68.500000
25%,2019.000000,351.250000,79.850000,82.450000
50%,2020.500000,400.000000,87.800000,88.700000
75%,2022.000000,427.500000,92.300000,93.550000
max,2023.000000,480.000000,9999.000000,97.200000


In [13]:
# ⑥ 컬럼명 & 자료형
print(f"⑥-1 컬럼 목록:\n{df.columns.tolist()}")
print(f"\n⑥-2 자료형:\n{df.dtypes}")

⑥-1 컬럼 목록:
['이름', '부서', '계급', '입사연도', '급여', '훈련점수', '출석률', '입대일']

⑥-2 자료형:
이름       object
부서       object
계급       object
입사연도      int64
급여      float64
훈련점수    float64
출석률     float64
입대일      object
dtype: object


## 3-2. 결측값 개수 확인

In [10]:
# 컬럼별 결측값 개수
print("[ 컬럼별 결측값 개수 ]")
print(df.isnull().sum())

# 결측 비율(%)
print("\n[ 결측 비율(%) ]")
print((df.isnull().sum() / len(df) * 100).round(2))

[ 컬럼별 결측값 개수 ]
이름      0
부서      0
계급      0
입사연도    0
급여      2
훈련점수    1
출석률     1
입대일     0
dtype: int64

[ 결측 비율(%) ]
이름       0.0
부서       0.0
계급       0.0
입사연도     0.0
급여      10.0
훈련점수     5.0
출석률      5.0
입대일      0.0
dtype: float64


## 3-3. 고유값(Unique Value) 확인

In [11]:
# 부서별 인원 수
print("[ 부서별 인원 수 ]")
print(df['부서'].value_counts())

# 고유값 목록
print(f"\n[ 계급 종류 ]: {df['계급'].unique()}")
print(f"[ 부서 종류 ]: {df['부서'].unique()}")
print(f"[ 계급 개수 ]: {df['계급'].nunique()}개")

[ 부서별 인원 수 ]
부서
행정과    7
통신과    7
보급과    6
Name: count, dtype: int64

[ 계급 종류 ]: ['상병' '병장' '이병' '일병']
[ 부서 종류 ]: ['행정과' '보급과' '통신과']
[ 계급 개수 ]: 4개


---
### 🔥 실습문제 3
`df` 데이터에 대해 다음을 수행하세요.

1. 전체 행 수, 열 수를 한 줄로 출력 (예: `데이터: 20행 × 8열`)
2. `"입사연도"` 컬럼의 **최솟값과 최댓값** 출력
3. `"출석률"` 컬럼의 **평균값** 을 소수점 둘째 자리까지 출력
4. `"부서"` 컬럼에서 **각 부서별 인원 수** 출력

In [ ]:
# ✍️ 여기에 코드를 작성하세요



**✅ 정답 예시**

In [21]:
# 1) 데이터 크기
print(f"데이터: {df.shape[0]}행 × {df.shape[1]}열")

# 2) 입사연도 범위
print(f"입사연도: {df['입사연도'].min()} ~ {df['입사연도'].max()}")

# 3) 출석률 평균
print(f"출석률 평균: {df['출석률'].mean():.2f}%")

# 4) 부서별 인원
print("\n부서별 인원 수:")
print(df['부서'].value_counts())

데이터: 20행 × 8열
입사연도: 2018 ~ 2023
출석률 평균: 86.83%

부서별 인원 수:
부서
행정과    7
통신과    7
보급과    6
Name: count, dtype: int64


---
# 🎯 Chapter 4. 데이터 선택 & 필터링

원하는 행·열만 골라내는 방법을 배웁니다.

## 4-1. 컬럼 선택

In [22]:
df = pd.read_csv('/content/soldiers.csv')

# 단일 컬럼 → Series
print("[ 단일 컬럼 (Series) ]")
print(df['급여'].head())
print(f"타입: {type(df['급여'])}")

[ 단일 컬럼 (Series) ]
0    420.0
1    380.0
2      NaN
3    450.0
4    410.0
Name: 급여, dtype: float64
타입: <class 'pandas.core.series.Series'>


In [23]:
# 여러 컬럼 → DataFrame (대괄호 2개 주의!)
print("[ 여러 컬럼 (DataFrame) ]")
print(df[['이름', '계급', '급여']].head())

[ 여러 컬럼 (DataFrame) ]
    이름  계급     급여
0  김철수  상병  420.0
1  이영희  병장  380.0
2  박민준  이병    NaN
3  최수진  일병  450.0
4  정태양  상병  410.0


## 4-2. `loc` (라벨 기반) vs `iloc` (위치 기반)

> **loc = 이름으로 / iloc = 번호로**

In [24]:
# loc: 라벨(이름) 기반
print("[ df.loc[0, '이름'] ]")
print(df.loc[0, '이름'])

print("\n[ df.loc[0:2, ['이름', '급여']] ]")
print(df.loc[0:2, ['이름', '급여']])

# iloc: 위치(정수 index) 기반
print("\n[ df.iloc[0:3, 0:3] ]")
print(df.iloc[0:3, 0:3])

[ df.loc[0, '이름'] ]
김철수

[ df.loc[0:2, ['이름', '급여']] ]
    이름     급여
0  김철수  420.0
1  이영희  380.0
2  박민준    NaN

[ df.iloc[0:3, 0:3] ]
    이름   부서  계급
0  김철수  행정과  상병
1  이영희  보급과  병장
2  박민준  통신과  이병


## 4-3. Boolean Indexing (조건 필터링)

In [ ]:
# 급여 400 이상
print("[ 급여 >= 400 ]")
high_pay = df[df['급여'] >= 400]
print(f"인원: {len(high_pay)}명")
high_pay[['이름', '급여']].head()

In [ ]:
# AND 조건: 급여 ≥ 400 AND 출석률 ≥ 90
# 주의: 각 조건을 괄호로 묶기! & 사용 (and 아님)
high = df[(df['급여'] >= 400) & (df['출석률'] >= 90)]
print(f"고급여+성실 인원: {len(high)}명")
high[['이름', '급여', '출석률']]

In [ ]:
# OR 조건
rank_select = df[(df['계급'] == '병장') | (df['계급'] == '상병')]
print(f"병장+상병: {len(rank_select)}명")

# isin(): 특정 값 포함
dept = df[df['부서'].isin(['행정과', '보급과'])]
print(f"행정과+보급과: {len(dept)}명")

# str.contains(): 문자열 포함 검색
kim = df[df['이름'].str.contains('김')]
print(f"'김'씨 인원: {len(kim)}명")
kim[['이름', '부서']]

---
### 🔥 실습문제 4
`df`에서 다음 조건을 만족하는 데이터를 추출하세요.

1. **"통신과"** 에 소속된 부대원 전체 출력
2. 훈련점수가 **90점 이상** 인 부대원의 `이름`, `계급`, `훈련점수` 출력
3. 계급이 **"상병" 또는 "병장"** 이면서 **출석률이 90 이상** 인 부대원 수 출력
4. 이름에 **"이"** 가 포함된 부대원 출력

In [ ]:
# ✍️ 여기에 코드를 작성하세요



**✅ 정답 예시**

In [ ]:
# 1) 통신과 부대원
print("[ 1. 통신과 부대원 ]")
print(df[df['부서'] == '통신과'])

# 2) 훈련점수 90 이상
print("\n[ 2. 훈련점수 90 이상 ]")
print(df[df['훈련점수'] >= 90][['이름', '계급', '훈련점수']])

# 3) 상병/병장 AND 출석률 90 이상
result3 = df[
    (df['계급'].isin(['상병', '병장'])) &
    (df['출석률'] >= 90)
]
print(f"\n[ 3. 상병/병장 + 출석률 90 이상 ]: {len(result3)}명")
print(result3[['이름', '계급', '출석률']])

# 4) 이름에 '이' 포함
print("\n[ 4. '이' 포함 ]")
print(df[df['이름'].str.contains('이')][['이름', '부서']])

---
# 🕳️ Chapter 5. 결측값(Missing Value) 처리

> ⚠️ **결측값 = 데이터가 비어 있는 값 (NaN, None, 빈칸)**  
> 분석 오류·왜곡의 원인이 되므로 반드시 처리해야 합니다.

## 5-1. 결측값 처리 전략 선택 가이드

| 결측 비율 | 권장 전략 | 예시 |
|---|---|---|
| **< 5%** | 행 삭제 (`dropna`) | 훈련점수 1% 결측 |
| **5~30%** | 평균/중앙값 채우기 | 체력점수 10% 결측 |
| **> 30%** | 열 삭제 고려 | 혈액형 36% 결측 |
| **범주형** | 최빈값(mode)으로 채우기 | 계급 결측 → '하사' |
| **시계열** | 앞/뒤 값(ffill/bfill) | 일별 기온 3일 결측 |

## 5-2. 결측값 확인

In [ ]:
df = pd.read_csv('/content/soldiers.csv')

# 결측값 개수
print("[ 컬럼별 결측값 개수 ]")
print(df.isnull().sum())

# 결측 비율
print("\n[ 결측 비율(%) ]")
print((df.isnull().sum() / len(df) * 100).round(2))

In [ ]:
# 결측값이 있는 행 확인
print("[ 급여가 결측인 행 ]")
print(df[df['급여'].isnull()])

## 5-3. 결측값 삭제 — `dropna()`

In [ ]:
# 원본 보존용 복사
df1 = df.copy()
print(f"삭제 전: {df1.shape}")

# 방법 1: 결측 있는 행 전체 삭제
df_drop_all = df1.dropna()
print(f"모든 결측 행 삭제: {df_drop_all.shape}")

# 방법 2: 특정 컬럼 결측인 행만 삭제
df_drop_pay = df1.dropna(subset=['급여'])
print(f"급여 결측 행만 삭제: {df_drop_pay.shape}")

# 방법 3: 모든 값이 결측인 행만 삭제
# df1.dropna(how='all')

# 방법 4: 임계값 (유효값 n개 미만인 행 삭제)
# df1.dropna(thresh=6)

## 5-4. 결측값 채우기 — `fillna()`

In [ ]:
df2 = df.copy()

# 방법 1: 고정값으로 채우기
df2_zero = df2['급여'].fillna(0)
print(f"[ 급여를 0으로 채우기 ]\n   2번 행 급여: {df2_zero[2]}")

# 방법 2: 평균값으로 채우기 (수치형 권장)
mean_pay = df2['급여'].mean()
df2['급여'] = df2['급여'].fillna(mean_pay)
print(f"\n[ 평균값 ({mean_pay:.1f})으로 채우기 완료 ]")
print(df2[['이름', '급여']].iloc[:5])

In [ ]:
df3 = df.copy()

# 방법 3: 중앙값으로 채우기 (이상치에 덜 민감)
df3['출석률'] = df3['출석률'].fillna(df3['출석률'].median())
print(f"출석률 중앙값으로 채우기 완료, 결측값: {df3['출석률'].isnull().sum()}개")

# 방법 4: 최빈값으로 채우기 (범주형)
# most = df3['계급'].mode()[0]
# df3['계급'] = df3['계급'].fillna(most)

# 방법 5: 앞/뒤 값으로 채우기 (시계열)
# df3.fillna(method='ffill')    # forward fill
# df3.fillna(method='bfill')    # backward fill

---
### 🔥 실습문제 5
다음 순서대로 결측값을 처리하세요.

1. `df_raw = pd.read_csv(...)`로 원본 다시 읽기
2. 결측값 **비율(%)** 확인하여 출력
3. `"급여"` 컬럼의 결측값을 **평균값**으로 채우기
4. `"훈련점수"` 컬럼의 결측값을 **중앙값**으로 채우기
5. `"출석률"` 컬럼이 결측인 **행을 삭제**
6. 처리 후 `isnull().sum()` 으로 결측값이 모두 제거되었는지 확인

In [ ]:
# ✍️ 여기에 코드를 작성하세요



**✅ 정답 예시**

In [ ]:
# 1) 원본 다시 읽기
df_raw = pd.read_csv('/content/soldiers.csv')
print(f"원본: {df_raw.shape}")

# 2) 결측값 비율
print("\n[ 처리 전 결측값 비율(%) ]")
print((df_raw.isnull().sum() / len(df_raw) * 100).round(2))

# 3) 급여 → 평균
df_raw['급여'] = df_raw['급여'].fillna(df_raw['급여'].mean())

# 4) 훈련점수 → 중앙값
df_raw['훈련점수'] = df_raw['훈련점수'].fillna(df_raw['훈련점수'].median())

# 5) 출석률 결측 행 삭제
df_raw = df_raw.dropna(subset=['출석률'])

# 6) 최종 확인
print(f"\n처리 후: {df_raw.shape}")
print("\n[ 처리 후 결측값 개수 ]")
print(df_raw.isnull().sum())

---
# 🔧 Chapter 6. 중복 제거 · 타입 변환 · 정규화

## 6-1. 중복 행 처리

In [25]:
df = pd.read_csv('/content/soldiers.csv')

# 중복 여부 확인
print(f"중복된 행 수: {df.duplicated().sum()}개")

# 중복 행 보기 (동명이인 '한소희' 케이스)
print("\n[ 이름 기준 중복 확인 ]")
dup_names = df[df['이름'].duplicated(keep=False)].sort_values('이름')
print(dup_names[['이름', '부서', '계급', '입대일']])

중복된 행 수: 0개

[ 이름 기준 중복 확인 ]
     이름   부서  계급         입대일
5   한소희  통신과  병장  2022-02-28
15  한소희  통신과  병장  2022-02-28


In [26]:
# 중복 제거 (기본: 첫 번째 행 유지)
df_nodup = df.drop_duplicates()
print(f"전체 컬럼 기준 제거: {df.shape} → {df_nodup.shape}")

# 특정 컬럼 기준 중복 제거
df_nodup_name = df.drop_duplicates(subset=['이름'])
print(f"이름 기준 제거: {df.shape} → {df_nodup_name.shape}")

# keep 옵션
# keep='first' (기본) - 처음 것만 유지
# keep='last'         - 마지막 것만 유지
# keep=False          - 중복된 것 모두 삭제

전체 컬럼 기준 제거: (20, 8) → (20, 8)
이름 기준 제거: (20, 8) → (19, 8)


## 6-2. 데이터 타입 변환 — `astype()`

In [27]:
df = pd.read_csv('/content/soldiers.csv')

# 변환 전 타입 확인
print("[ 변환 전 dtypes ]")
print(df.dtypes)

[ 변환 전 dtypes ]
이름       object
부서       object
계급       object
입사연도      int64
급여      float64
훈련점수    float64
출석률     float64
입대일      object
dtype: object


In [28]:
# 문자열 → 날짜 (가장 중요한 변환)
df['입대일'] = pd.to_datetime(df['입대일'])

# 연/월/일 추출 가능
df['입대연도'] = df['입대일'].dt.year
df['입대월']   = df['입대일'].dt.month
df['요일']     = df['입대일'].dt.day_name()

print("[ 날짜 변환 후 ]")
print(df[['이름', '입대일', '입대연도', '입대월', '요일']].head())
print(f"\n입대일 타입: {df['입대일'].dtype}")

[ 날짜 변환 후 ]
    이름        입대일  입대연도  입대월        요일
0  김철수 2019-03-01  2019    3    Friday
1  이영희 2021-06-15  2021    6   Tuesday
2  박민준 2020-09-10  2020    9  Thursday
3  최수진 2018-04-20  2018    4    Friday
4  정태양 2019-07-05  2019    7    Friday

입대일 타입: datetime64[ns]


In [ ]:
# 범주형으로 변환 (메모리 절약)
df['계급'] = df['계급'].astype('category')
print(f"계급 타입: {df['계급'].dtype}")
print(f"카테고리: {df['계급'].cat.categories.tolist()}")

## 6-3. 정규화 (Normalization) & 표준화 (Standardization)

| 방법 | 수식 | 결과 범위 | 언제? |
|---|---|---|---|
| **Min-Max 정규화** | (x - min) / (max - min) | 0 ~ 1 | 범위 통일 |
| **표준화 (Z-score)** | (x - 평균) / 표준편차 | 평균=0, 표준편차=1 | 이상치 영향 최소화 |

In [29]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# 원본 데이터 준비 (이상치 9999 제거 후)
df_clean = df[df['훈련점수'] < 1000].copy().reset_index(drop=True)

# ① Min-Max 정규화 (0~1)
mm = MinMaxScaler()
df_clean['훈련점수_정규화'] = mm.fit_transform(df_clean[['훈련점수']])

# ② 표준화 (Z-score)
std = StandardScaler()
df_clean['훈련점수_표준화'] = std.fit_transform(df_clean[['훈련점수']])

print(df_clean[['이름', '훈련점수', '훈련점수_정규화', '훈련점수_표준화']].head(10).round(3))

print(f"\n정규화 범위 : {df_clean['훈련점수_정규화'].min():.3f} ~ {df_clean['훈련점수_정규화'].max():.3f}")
print(f"표준화 평균 : {df_clean['훈련점수_표준화'].mean():.3f}")
print(f"표준화 표편 : {df_clean['훈련점수_표준화'].std():.3f}")

    이름  훈련점수  훈련점수_정규화  훈련점수_표준화
0  김철수  85.2     0.598    -0.001
1  이영희  92.3     0.886     0.942
2  박민준  78.5     0.325    -0.892
3  최수진  95.1     1.000     1.314
4  정태양  88.0     0.711     0.371
5  윤도현  72.1     0.065    -1.743
6  강민아  80.5     0.407    -0.626
7  송지훈  91.2     0.841     0.796
8  임서연  87.8     0.703     0.344
9  오현우  75.0     0.183    -1.357

정규화 범위 : 0.000 ~ 1.000
표준화 평균 : 0.000
표준화 표편 : 1.029


## 6-4. 범주형 데이터 인코딩

In [30]:
df_enc = pd.read_csv('/content/soldiers.csv')

# 방법 1: map() - 순서가 있는 범주형 (ordinal)
rank_map = {'이병': 1, '일병': 2, '상병': 3, '병장': 4}
df_enc['계급코드'] = df_enc['계급'].map(rank_map)

print("[ map()을 이용한 레이블 인코딩 ]")
print(df_enc[['이름', '계급', '계급코드']].head())

[ map()을 이용한 레이블 인코딩 ]
    이름  계급  계급코드
0  김철수  상병     3
1  이영희  병장     4
2  박민준  이병     1
3  최수진  일병     2
4  정태양  상병     3


In [31]:
# 방법 2: pd.get_dummies() - 원-핫 인코딩 (순서 없는 범주형)
df_dummies = pd.get_dummies(df_enc[['이름', '부서']], columns=['부서'])
print("[ 원-핫 인코딩 ]")
print(df_dummies.head())

[ 원-핫 인코딩 ]
    이름  부서_보급과  부서_통신과  부서_행정과
0  김철수   False   False    True
1  이영희    True   False   False
2  박민준   False    True   False
3  최수진   False   False    True
4  정태양    True   False   False


---
### 🔥 실습문제 6
`df = pd.read_csv('/content/soldiers.csv')` 를 원본으로 다음을 수행하세요.

1. **이름 기준** 중복 행을 제거 (처음 발견된 행만 유지)
2. `"입대일"` 컬럼을 **datetime** 타입으로 변환
3. `"입대요일"` 이라는 새 컬럼을 만들고 **요일(한글, 예: '월요일')** 채우기
4. `"급여"` 컬럼을 **Min-Max 정규화**하여 `"급여_정규화"` 컬럼 추가
5. 결과 5행 출력

In [ ]:
# ✍️ 여기에 코드를 작성하세요



**✅ 정답 예시**

In [32]:
from sklearn.preprocessing import MinMaxScaler

df = pd.read_csv('/content/soldiers.csv')

# 1) 이름 기준 중복 제거
df = df.drop_duplicates(subset=['이름'], keep='first').reset_index(drop=True)
print(f"중복 제거 후: {df.shape}")

# 2) 입대일 → datetime
df['입대일'] = pd.to_datetime(df['입대일'])

# 3) 요일 컬럼 (한글)
weekday_kr = {0: '월요일', 1: '화요일', 2: '수요일',
              3: '목요일', 4: '금요일', 5: '토요일', 6: '일요일'}
df['입대요일'] = df['입대일'].dt.weekday.map(weekday_kr)

# 4) 급여 정규화 (결측값은 먼저 평균으로 채운 후 정규화)
df['급여'] = df['급여'].fillna(df['급여'].mean())
mm = MinMaxScaler()
df['급여_정규화'] = mm.fit_transform(df[['급여']])

# 5) 결과 확인
df[['이름', '입대일', '입대요일', '급여', '급여_정규화']].head().round(3)

중복 제거 후: (19, 8)


,이름,입대일,입대요일,급여,급여_정규화
0,김철수,2019-03-01,금요일,420.000,0.739
1,이영희,2021-06-15,화요일,380.000,0.565
2,박민준,2020-09-10,목요일,385.294,0.588
3,최수진,2018-04-20,금요일,450.000,0.870
4,정태양,2019-07-05,금요일,410.000,0.696


---
# 📊 Chapter 7. 이상치(Outlier) 탐지 및 처리

> ⚠️ **이상치** = 다른 데이터와 크게 벗어난 값  
> 예: 급여가 0원, 나이가 200세, 출석률이 150%, 훈련점수가 9999점 등

## 7-1. `describe()` 로 분포 확인

In [33]:
df = pd.read_csv('/content/soldiers.csv')

# 수치형 컬럼 통계
print(df[['급여', '훈련점수', '출석률']].describe().round(2))

# 🔍 훈련점수 max가 9999로 나타남 → 이상치!
print(f"\n훈련점수 min: {df['훈련점수'].min()}")
print(f"훈련점수 max: {df['훈련점수'].max()}   ← 이상치 의심")

           급여     훈련점수    출석률
count   18.00    19.00  19.00
mean   385.56   606.99  86.83
std     72.58  2274.39   8.42
min    250.00    70.50  68.50
25%    351.25    79.85  82.45
50%    400.00    87.80  88.70
75%    427.50    92.30  93.55
max    480.00  9999.00  97.20

훈련점수 min: 70.5
훈련점수 max: 9999.0   ← 이상치 의심


## 7-2. IQR 방법 (실무에서 가장 많이 씀)

```
IQR = Q3 - Q1   (사분위 범위)
이상치 범위: [Q1 - 1.5*IQR, Q3 + 1.5*IQR]
이 범위를 벗어나면 이상치로 판정
```

In [34]:
# 훈련점수에서 이상치 탐지
col = '훈련점수'

Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"Q1       : {Q1:.2f}")
print(f"Q3       : {Q3:.2f}")
print(f"IQR      : {IQR:.2f}")
print(f"하한값   : {lower:.2f}")
print(f"상한값   : {upper:.2f}")

# 이상치 추출
outliers = df[(df[col] < lower) | (df[col] > upper)]
print(f"\n⚠️ 이상치 {len(outliers)}개 발견")
print(outliers[['이름', col]])

Q1       : 79.85
Q3       : 92.30
IQR      : 12.45
하한값   : 61.17
상한값   : 110.97

⚠️ 이상치 1개 발견
     이름    훈련점수
14  문재혁  9999.0


## 7-3. 이상치 처리 방법

In [35]:
# 방법 1: 이상치 제거
df_clean = df[(df[col] >= lower) & (df[col] <= upper)].copy()
print(f"제거 전: {df.shape[0]}행 → 제거 후: {df_clean.shape[0]}행")

# 방법 2: 이상치를 NaN으로 치환 후 평균으로 채우기
df_fix = df.copy()
df_fix.loc[(df_fix[col] < lower) | (df_fix[col] > upper), col] = np.nan
df_fix[col] = df_fix[col].fillna(df_fix[col].mean())

print(f"\n방법 2 결과 (이상치를 평균으로 대체):")
print(df_fix[['이름', col]].iloc[[14]])   # 이상치였던 행

# 방법 3: Capping (경계값으로 변경)
df_cap = df.copy()
df_cap[col] = df_cap[col].clip(lower=lower, upper=upper)
print(f"\n방법 3 결과 (경계값으로 Capping):")
print(df_cap[['이름', col]].iloc[[14]])

제거 전: 20행 → 제거 후: 18행


NameError: name 'np' is not defined

## 7-4. 박스플롯(Boxplot) 시각화

In [36]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 이상치 포함 (원본)
axes[0].boxplot(df['훈련점수'].dropna())
axes[0].set_title("처리 전 (이상치 포함)", fontsize=12)
axes[0].set_ylabel("훈련점수")

# 이상치 제거 후
axes[1].boxplot(df_clean['훈련점수'].dropna())
axes[1].set_title("처리 후 (이상치 제거)", fontsize=12)
axes[1].set_ylabel("훈련점수")

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

---
### 🔥 실습문제 7
`"급여"` 컬럼에서 **IQR 방법**으로 이상치를 탐지하고 처리하세요.

1. 결측값을 평균값으로 먼저 채우기
2. Q1, Q3, IQR 계산
3. 이상치의 상·하한선 출력
4. 이상치 **개수** 확인
5. 이상치를 **제거한** 새 DataFrame 생성

In [ ]:
# ✍️ 여기에 코드를 작성하세요



**✅ 정답 예시**

In [37]:
df = pd.read_csv('/content/soldiers.csv')

# 1) 결측값 평균으로 채우기
df['급여'] = df['급여'].fillna(df['급여'].mean())

# 2) IQR 계산
col = '급여'
Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

# 3) 범위 출력
print(f"Q1 = {Q1:.1f}, Q3 = {Q3:.1f}, IQR = {IQR:.1f}")
print(f"정상 범위: [{lower:.1f}, {upper:.1f}]")

# 4) 이상치 개수
out_mask = (df[col] < lower) | (df[col] > upper)
print(f"이상치 개수: {out_mask.sum()}개")
if out_mask.sum() > 0:
    print(df[out_mask][['이름', '급여']])

# 5) 이상치 제거
df_clean = df[~out_mask].reset_index(drop=True)
print(f"\n제거 후: {df.shape[0]} → {df_clean.shape[0]}행")

Q1 = 353.8, Q3 = 422.5, IQR = 68.8
정상 범위: [250.6, 525.6]
이상치 개수: 1개
    이름     급여
6  윤도현  250.0

제거 후: 20 → 19행


---
# 📈 Chapter 8. 데이터 그룹화 & 집계 — `groupby`

> 실무에서 **가장 자주 쓰는** 분석: 부서별 평균, 월별 합계, 계급별 통계 등

## 8-1. GroupBy 동작 원리: Split → Apply → Combine

```
   ✂️ Split          ⚙️ Apply        🔗 Combine
   ─────────         ────────       ──────────
   그룹 기준으로  →  각 그룹에    →  결과를 합쳐
   데이터 분리       집계 함수 적용    새 DataFrame 생성
```

In [38]:
# 실습용 깨끗한 데이터 준비
df = pd.read_csv('/content/soldiers.csv')
df['급여'] = df['급여'].fillna(df['급여'].mean())
df['훈련점수'] = df['훈련점수'].replace(9999, np.nan).fillna(df['훈련점수'][df['훈련점수']<1000].mean())
df['출석률'] = df['출석률'].fillna(df['출석률'].median())
df = df.drop_duplicates(subset=['이름'])
df.head(3)

NameError: name 'np' is not defined

## 8-2. 단일 컬럼 집계

In [39]:
# 부서별 평균 급여
print("[ 부서별 평균 급여 ]")
print(df.groupby('부서')['급여'].mean().round(1))

# 부서별 인원 수
print("\n[ 부서별 인원 수 ]")
print(df.groupby('부서').size())

# 계급별 평균 훈련점수
print("\n[ 계급별 평균 훈련점수 ]")
print(df.groupby('계급')['훈련점수'].mean().round(2))

[ 부서별 평균 급여 ]
부서
보급과    372.5
통신과    380.8
행정과    401.5
Name: 급여, dtype: float64

[ 부서별 인원 수 ]
부서
보급과    6
통신과    7
행정과    7
dtype: int64

[ 계급별 평균 훈련점수 ]
계급
병장      91.96
상병      88.14
이병    2059.02
일병      84.30
Name: 훈련점수, dtype: float64


## 8-3. 여러 집계 함수 동시 적용 — `agg()`

In [40]:
# 방법 1: 리스트로 여러 함수
result1 = df.groupby('부서')['급여'].agg(['mean', 'max', 'min', 'count'])
print("[ 부서별 급여 통계 ]")
print(result1.round(1))

[ 부서별 급여 통계 ]
      mean    max    min  count
부서                             
보급과  372.5  480.0  260.0      6
통신과  380.8  475.0  255.0      7
행정과  401.5  470.0  250.0      7


In [41]:
# 방법 2: 컬럼별로 다른 함수 지정
result2 = df.groupby('부서').agg({
    '급여': 'mean',
    '훈련점수': ['mean', 'max'],
    '출석률': 'mean'
}).round(2)

print("[ 부서별 다중 집계 ]")
print(result2)

[ 부서별 다중 집계 ]
         급여     훈련점수            출석률
       mean     mean     max   mean
부서                                 
보급과  372.50    84.88    94.3  87.70
통신과  380.79  1739.42  9999.0  82.86
행정과  401.51    83.86    95.1  90.60


In [42]:
# 방법 3: 가독성 좋게 컬럼명 직접 지정
result3 = df.groupby('부서').agg(
    평균급여    = ('급여', 'mean'),
    최대점수    = ('훈련점수', 'max'),
    평균출석률  = ('출석률', 'mean'),
    인원수      = ('이름', 'count')
).round(1).reset_index()

print("[ 이름 지정 집계 (가독성 good) ]")
print(result3)

[ 이름 지정 집계 (가독성 good) ]
    부서   평균급여    최대점수  평균출석률  인원수
0  보급과  372.5    94.3   87.7    6
1  통신과  380.8  9999.0   82.9    7
2  행정과  401.5    95.1   90.6    7


In [43]:
{
 "nbformat": 4,
 "nbformat_minor": 0,
 "metadata": {
  "colab": {
   "provenance": [],
   "name": "AI_여행플래너_기상연동.ipynb"
  },
  "kernelspec": {
   "name": "python3",
   "display_name": "Python 3"
  },
  "language_info": {
   "name": "python"
  }
 },
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 🗺️ AI 여행 플래너 — 기상청 날씨 연동\n",
    "> 도시 입력 → 여행 기간 설정 → 조건 선택 → AI가 날씨 기반 일정 생성\n",
    "\n",
    "**필요한 API 키**\n",
    "- [기상청 Open API](https://www.data.go.kr/) 에서 `중기예보 API` 키 발급\n",
    "- [Anthropic API](https://console.anthropic.com/) 에서 Claude API 키 발급"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ────────────────────────────────────────────\n",
    "# CELL 1 : 라이브러리 설치\n",
    "# ────────────────────────────────────────────\n",
    "!pip install anthropic requests -q"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ────────────────────────────────────────────\n",
    "# CELL 2 : API 키 설정\n",
    "# ────────────────────────────────────────────\n",
    "import anthropic\n",
    "import requests\n",
    "import json\n",
    "from datetime import datetime, timedelta\n",
    "from IPython.display import display, HTML\n",
    "import ipywidgets as widgets\n",
    "\n",
    "# ⚠️ 아래 두 키를 본인 키로 교체하세요\n",
    "KMA_API_KEY   = \"ab8817dc5f825af6b9573f9b8b87f41ed03db0fce3fc0ac6ce64064d65deb29a\"   # 공공데이터포털 기상청 API 키\n",
    "CLAUDE_API_KEY = \"sk-ant-...\"           # Anthropic Claude API 키\n",
    "\n",
    "client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)\n",
    "print(\"✅ API 클라이언트 초기화 완료\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ────────────────────────────────────────────\n",
    "# CELL 3 : 기상청 API 설정 — 도시별 코드\n",
    "# ────────────────────────────────────────────\n",
    "\n",
    "# 기상청 중기예보 지점 코드 (regId)\n",
    "CITY_CODE = {\n",
    "    \"서울\":    {\"reg\": \"11B00000\", \"tm\": \"108\"},\n",
    "    \"인천\":    {\"reg\": \"11B00000\", \"tm\": \"112\"},\n",
    "    \"경기\":    {\"reg\": \"11B00000\", \"tm\": \"119\"},\n",
    "    \"파주\":    {\"reg\": \"11B00000\", \"tm\": \"119\"},\n",
    "    \"강원\":    {\"reg\": \"11D10000\", \"tm\": \"100\"},\n",
    "    \"춘천\":    {\"reg\": \"11D10000\", \"tm\": \"101\"},\n",
    "    \"강릉\":    {\"reg\": \"11D20000\", \"tm\": \"105\"},\n",
    "    \"대전\":    {\"reg\": \"11C20000\", \"tm\": \"133\"},\n",
    "    \"세종\":    {\"reg\": \"11C20000\", \"tm\": \"133\"},\n",
    "    \"충북\":    {\"reg\": \"11C10000\", \"tm\": \"131\"},\n",
    "    \"충남\":    {\"reg\": \"11C20000\", \"tm\": \"232\"},\n",
    "    \"광주\":    {\"reg\": \"11F20000\", \"tm\": \"156\"},\n",
    "    \"전북\":    {\"reg\": \"11F10000\", \"tm\": \"146\"},\n",
    "    \"전남\":    {\"reg\": \"11F20000\", \"tm\": \"165\"},\n",
    "    \"대구\":    {\"reg\": \"11H10000\", \"tm\": \"143\"},\n",
    "    \"경북\":    {\"reg\": \"11H10000\", \"tm\": \"136\"},\n",
    "    \"부산\":    {\"reg\": \"11H20000\", \"tm\": \"159\"},\n",
    "    \"울산\":    {\"reg\": \"11H20000\", \"tm\": \"152\"},\n",
    "    \"경남\":    {\"reg\": \"11H20000\", \"tm\": \"155\"},\n",
    "    \"제주\":    {\"reg\": \"11G00000\", \"tm\": \"184\"},\n",
    "}\n",
    "\n",
    "# 날씨 아이콘 매핑\n",
    "WEATHER_ICON = {\n",
    "    \"맑음\":        \"☀️\",\n",
    "    \"구름많음\":    \"⛅\",\n",
    "    \"흐림\":        \"☁️\",\n",
    "    \"비\":          \"🌧️\",\n",
    "    \"눈\":          \"❄️\",\n",
    "    \"비또는눈\":    \"🌨️\",\n",
    "}\n",
    "\n",
    "print(\"✅ 도시 코드 로드 완료 —\", list(CITY_CODE.keys()))"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ────────────────────────────────────────────\n",
    "# CELL 4 : 기상청 API 호출 함수\n",
    "# ────────────────────────────────────────────\n",
    "\n",
    "def get_midterm_weather(city: str, start_date: datetime, days: int) -> list[dict]:\n",
    "    \"\"\"\n",
    "    기상청 중기예보 API로 날씨 조회.\n",
    "    중기예보는 3~10일 후만 제공 → 3일 이내는 단기예보로 대체.\n",
    "    반환: [{date, weather, min_temp, max_temp, rain_prob}, ...]\n",
    "    \"\"\"\n",
    "    if city not in CITY_CODE:\n",
    "        # 입력 도시가 코드에 없으면 가장 유사한 광역 코드 매핑\n",
    "        for key in CITY_CODE:\n",
    "            if key in city or city in key:\n",
    "                city = key\n",
    "                break\n",
    "        else:\n",
    "            city = \"서울\"  # 기본값\n",
    "\n",
    "    reg_id = CITY_CODE[city][\"reg\"]\n",
    "    tm_id  = CITY_CODE[city][\"tm\"]\n",
    "\n",
    "    # 발표 시각: 06시 or 18시 기준\n",
    "    now = datetime.now()\n",
    "    if now.hour < 18:\n",
    "        tmFc = now.strftime(\"%Y%m%d\") + \"0600\"\n",
    "    else:\n",
    "        tmFc = now.strftime(\"%Y%m%d\") + \"1800\"\n",
    "\n",
    "    # ① 중기 날씨 (육상예보)\n",
    "    land_url = \"https://apis.data.go.kr/1360000/MidFcstInfoService/getMidLandFcst\"\n",
    "    land_params = {\n",
    "        \"serviceKey\": KMA_API_KEY,\n",
    "        \"pageNo\": 1, \"numOfRows\": 10,\n",
    "        \"dataType\": \"JSON\",\n",
    "        \"regId\": reg_id,\n",
    "        \"tmFc\": tmFc,\n",
    "    }\n",
    "\n",
    "    # ② 중기 기온\n",
    "    temp_url = \"https://apis.data.go.kr/1360000/MidFcstInfoService/getMidTa\"\n",
    "    temp_params = {\n",
    "        \"serviceKey\": KMA_API_KEY,\n",
    "        \"pageNo\": 1, \"numOfRows\": 10,\n",
    "        \"dataType\": \"JSON\",\n",
    "        \"regId\": tm_id,\n",
    "        \"tmFc\": tmFc,\n",
    "    }\n",
    "\n",
    "    results = []\n",
    "    try:\n",
    "        land_res = requests.get(land_url, params=land_params, timeout=10)\n",
    "        temp_res = requests.get(temp_url, params=temp_params, timeout=10)\n",
    "        land_data = land_res.json()[\"response\"][\"body\"][\"items\"][\"item\"][0]\n",
    "        temp_data = temp_res.json()[\"response\"][\"body\"][\"items\"][\"item\"][0]\n",
    "\n",
    "        for d in range(3, min(days + 3, 11)):\n",
    "            day_key = f\"wf{d}Am\" if d <= 7 else f\"wf{d}\"\n",
    "            rnSt_key = f\"rnSt{d}Am\" if d <= 7 else f\"rnSt{d}\"\n",
    "            min_key = f\"taMin{d}\"\n",
    "            max_key = f\"taMax{d}\"\n",
    "\n",
    "            weather_str = land_data.get(day_key, \"정보없음\")\n",
    "            results.append({\n",
    "                \"date\": (start_date + timedelta(days=d-3)).strftime(\"%Y-%m-%d\"),\n",
    "                \"day_label\": (start_date + timedelta(days=d-3)).strftime(\"%m/%d (%a)\"),\n",
    "                \"weather\": weather_str,\n",
    "                \"icon\": WEATHER_ICON.get(weather_str, \"🌤️\"),\n",
    "                \"min_temp\": temp_data.get(min_key, \"?\"),\n",
    "                \"max_temp\": temp_data.get(max_key, \"?\"),\n",
    "                \"rain_prob\": land_data.get(rnSt_key, \"?\"),\n",
    "            })\n",
    "            if len(results) >= days:\n",
    "                break\n",
    "\n",
    "    except Exception as e:\n",
    "        print(f\"⚠️ 기상청 API 오류: {e}\")\n",
    "        print(\"   → 샘플 날씨 데이터로 대체합니다.\")\n",
    "        # API 키 미설정 시 샘플 데이터\n",
    "        sample_weathers = [\"맑음\", \"구름많음\", \"흐림\", \"비\", \"맑음\"]\n",
    "        for d in range(days):\n",
    "            w = sample_weathers[d % len(sample_weathers)]\n",
    "            results.append({\n",
    "                \"date\": (start_date + timedelta(days=d)).strftime(\"%Y-%m-%d\"),\n",
    "                \"day_label\": (start_date + timedelta(days=d)).strftime(\"%m/%d (%a)\"),\n",
    "                \"weather\": w,\n",
    "                \"icon\": WEATHER_ICON.get(w, \"🌤️\"),\n",
    "                \"min_temp\": 12 + d,\n",
    "                \"max_temp\": 22 + d,\n",
    "                \"rain_prob\": [10, 20, 60, 80, 10][d % 5],\n",
    "            })\n",
    "    return results\n",
    "\n",
    "\n",
    "print(\"✅ 기상청 API 함수 정의 완료\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ────────────────────────────────────────────\n",
    "# CELL 5 : Claude AI 여행 일정 생성 함수\n",
    "# ────────────────────────────────────────────\n",
    "\n",
    "def generate_travel_plan(\n",
    "    city: str,\n",
    "    weather_list: list[dict],\n",
    "    with_dog: bool,\n",
    "    has_car: bool,\n",
    ") -> str:\n",
    "    \"\"\"\n",
    "    Claude API로 날씨 + 조건에 맞는 여행 일정 생성.\n",
    "    \"\"\"\n",
    "    days = len(weather_list)\n",
    "\n",
    "    # 날씨 요약 텍스트\n",
    "    weather_summary = \"\\n\".join([\n",
    "        f\"  - {w['day_label']}: {w['icon']} {w['weather']}, \"\n",
    "        f\"최저 {w['min_temp']}°C / 최고 {w['max_temp']}°C, \"\n",
    "        f\"강수확률 {w['rain_prob']}%\"\n",
    "        for w in weather_list\n",
    "    ])\n",
    "\n",
    "    dog_condition = \"반려견 동반 여행 (애견 동반 가능 장소만 포함)\" if with_dog else \"반려견 없음\"\n",
    "    car_condition = \"자차 이용 가능 (드라이브 코스, 주차 고려)\" if has_car else \"대중교통 이용 (버스·지하철 접근 가능 장소 위주)\"\n",
    "\n",
    "    prompt = f\"\"\"당신은 한국 여행 전문 플래너입니다. 아래 조건으로 {days}일 여행 일정을 작성해주세요.\n",
    "\n",
    "【여행 정보】\n",
    "- 여행지: {city}\n",
    "- 기간: {weather_list[0]['date']} ~ {weather_list[-1]['date']} ({days}일)\n",
    "- 반려견: {dog_condition}\n",
    "- 교통: {car_condition}\n",
    "\n",
    "【기상청 날씨 예보】\n",
    "{weather_summary}\n",
    "\n",
    "【작성 규칙】\n",
    "1. 날씨에 맞게 일정을 조정하세요 (비 오는 날 → 실내 중심, 맑은 날 → 야외 중심)\n",
    "2. 강수확률 60% 이상인 날은 우산 챙기기 알림 포함\n",
    "3. 각 날짜마다: 오전 / 오후 / 저녁 활동을 구체적으로 제시\n",
    "4. 실제 존재하는 장소 이름 사용 (카페, 식당, 관광지)\n",
    "5. 반려견 동반 시 → 애견 동반 가능 여부를 (🐾 동반가능) 표시\n",
    "6. 자차 없을 시 → 각 장소 옆에 대중교통 접근법 간략히 표시\n",
    "7. 마지막에 여행 팁 3가지 추가\n",
    "\n",
    "아래 형식으로 작성:\n",
    "\n",
    "## 📅 N일차 (날짜) [날씨아이콘 날씨] 최저/최고°C\n",
    "### 🌅 오전\n",
    "- 장소명: 설명\n",
    "### 🌞 오후  \n",
    "- 장소명: 설명\n",
    "### 🌙 저녁\n",
    "- 식당명: 추천 메뉴\n",
    "\n",
    "---\n",
    "\"\"\"\n",
    "\n",
    "    message = client.messages.create(\n",
    "        model=\"claude-sonnet-4-20250514\",\n",
    "        max_tokens=4000,\n",
    "        messages=[{\"role\": \"user\", \"content\": prompt}]\n",
    "    )\n",
    "    return message.content[0].text\n",
    "\n",
    "\n",
    "print(\"✅ AI 여행 플래너 함수 정의 완료\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ────────────────────────────────────────────\n",
    "# CELL 6 : HTML 결과 렌더링 함수\n",
    "# ────────────────────────────────────────────\n",
    "import re\n",
    "\n",
    "def render_weather_cards(weather_list: list[dict], city: str) -> None:\n",
    "    \"\"\"날씨 카드 HTML 렌더링\"\"\"\n",
    "    cards = \"\"\n",
    "    for w in weather_list:\n",
    "        rain = int(w['rain_prob']) if str(w['rain_prob']).isdigit() else 0\n",
    "        rain_color = \"#e74c3c\" if rain >= 60 else \"#3498db\" if rain >= 30 else \"#27ae60\"\n",
    "        cards += f\"\"\"\n",
    "        <div style=\"background:#fff;border-radius:12px;padding:16px 12px;text-align:center;\n",
    "                    box-shadow:0 2px 8px rgba(0,0,0,.08);min-width:120px;flex:1\">\n",
    "          <div style=\"font-size:12px;color:#888;margin-bottom:4px\">{w['day_label']}</div>\n",
    "          <div style=\"font-size:28px;margin:6px 0\">{w['icon']}</div>\n",
    "          <div style=\"font-size:12px;font-weight:600;color:#444;margin-bottom:6px\">{w['weather']}</div>\n",
    "          <div style=\"font-size:13px;color:#2c3e50\">\n",
    "            <span style=\"color:#3498db\">{w['min_temp']}°</span>\n",
    "            <span style=\"color:#aaa;margin:0 2px\">/</span>\n",
    "            <span style=\"color:#e74c3c\">{w['max_temp']}°</span>\n",
    "          </div>\n",
    "          <div style=\"margin-top:6px;font-size:11px;color:{rain_color};font-weight:600\">\n",
    "            💧 {w['rain_prob']}%\n",
    "          </div>\n",
    "        </div>\"\"\"\n",
    "\n",
    "    html = f\"\"\"\n",
    "    <div style=\"font-family:'Apple SD Gothic Neo',sans-serif;margin:16px 0\">\n",
    "      <div style=\"background:linear-gradient(135deg,#667eea,#764ba2);border-radius:16px;\n",
    "                  padding:20px 24px;color:#fff;margin-bottom:16px\">\n",
    "        <div style=\"font-size:20px;font-weight:700\">🗺️ {city} 여행 날씨 예보</div>\n",
    "        <div style=\"font-size:13px;opacity:.8;margin-top:4px\">기상청 중기예보 기준</div>\n",
    "      </div>\n",
    "      <div style=\"display:flex;gap:10px;flex-wrap:wrap\">{cards}</div>\n",
    "    </div>\n",
    "    \"\"\"\n",
    "    display(HTML(html))\n",
    "\n",
    "\n",
    "def render_plan_html(plan_text: str) -> None:\n",
    "    \"\"\"마크다운 일정을 보기 좋은 HTML로 변환\"\"\"\n",
    "    # 간단한 마크다운 → HTML 변환\n",
    "    html_text = plan_text\n",
    "    html_text = re.sub(r'^## (.+)$', r'<h2 style=\"color:#5c35d4;border-left:4px solid #5c35d4;padding-left:12px;margin-top:24px\">\\1</h2>', html_text, flags=re.MULTILINE)\n",
    "    html_text = re.sub(r'^### (.+)$', r'<h3 style=\"color:#2980b9;margin:14px 0 6px\">\\1</h3>', html_text, flags=re.MULTILINE)\n",
    "    html_text = re.sub(r'^- (.+)$', r'<li style=\"margin:5px 0;line-height:1.6\">\\1</li>', html_text, flags=re.MULTILINE)\n",
    "    html_text = re.sub(r'^---$', r'<hr style=\"border:none;border-top:1px solid #eee;margin:20px 0\">', html_text, flags=re.MULTILINE)\n",
    "    html_text = re.sub(r'\\*\\*(.+?)\\*\\*', r'<strong>\\1</strong>', html_text)\n",
    "    html_text = html_text.replace('\\n', '<br>')\n",
    "\n",
    "    display(HTML(f\"\"\"\n",
    "    <div style=\"font-family:'Apple SD Gothic Neo',sans-serif;background:#fafafa;\n",
    "                border-radius:16px;padding:24px;margin-top:16px;\n",
    "                border:1px solid #eee;line-height:1.7;font-size:14px\">\n",
    "      {html_text}\n",
    "    </div>\n",
    "    \"\"\"))\n",
    "\n",
    "\n",
    "print(\"✅ 렌더링 함수 정의 완료\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# ────────────────────────────────────────────\n",
    "# CELL 7 : 🎯 메인 UI — 입력 위젯\n",
    "# ────────────────────────────────────────────\n",
    "\n",
    "# ── 스타일 헤더\n",
    "display(HTML(\"\"\"\n",
    "<div style=\"font-family:'Apple SD Gothic Neo',sans-serif;background:linear-gradient(135deg,#1a1a2e,#16213e);\n",
    "     border-radius:20px;padding:28px 32px;color:#fff;margin-bottom:20px\">\n",
    "  <div style=\"font-size:26px;font-weight:800;letter-spacing:-0.5px\">✈️ AI 여행 플래너</div>\n",
    "  <div style=\"font-size:14px;opacity:.7;margin-top:6px\">기상청 날씨 + Claude AI 일정 자동 생성</div>\n",
    "</div>\n",
    "\"\"\"))\n",
    "\n",
    "# ── 위젯 정의\n",
    "city_input = widgets.Text(\n",
    "    value=\"파주\",\n",
    "    placeholder=\"예: 서울, 제주, 부산, 강릉\",\n",
    "    description=\"📍 도시\",\n",
    "    style={\"description_width\": \"80px\"},\n",
    "    layout=widgets.Layout(width=\"360px\")\n",
    ")\n",
    "\n",
    "start_date_input = widgets.DatePicker(\n",
    "    description=\"📅 시작일\",\n",
    "    value=datetime.now().date() + timedelta(days=4),\n",
    "    style={\"description_width\": \"80px\"},\n",
    "    layout=widgets.Layout(width=\"260px\")\n",
    ")\n",
    "\n",
    "days_input = widgets.IntSlider(\n",
    "    value=3,\n",
    "    min=1, max=7, step=1,\n",
    "    description=\"🗓️ 기간\",\n",
    "    style={\"description_width\": \"80px\"},\n",
    "    layout=widgets.Layout(width=\"360px\"),\n",
    "    readout_format=\"d\",\n",
    ")\n",
    "days_label = widgets.Label(value=\"3 박 4 일\")\n",
    "\n",
    "def update_days_label(change):\n",
    "    n = change[\"new\"]\n",
    "    days_label.value = f\"{n} 박 {n+1} 일  →  총 {n+1}일 일정\"\n",
    "days_input.observe(update_days_label, names=\"value\")\n",
    "\n",
    "dog_toggle = widgets.ToggleButton(\n",
    "    value=False,\n",
    "    description=\"🐾 애견 동반\",\n",
    "    button_style=\"\",\n",
    "    layout=widgets.Layout(width=\"140px\", height=\"40px\")\n",
    ")\n",
    "\n",
    "car_toggle = widgets.ToggleButton(\n",
    "    value=True,\n",
    "    description=\"🚗 자차 이용\",\n",
    "    button_style=\"\",\n",
    "    layout=widgets.Layout(width=\"140px\", height=\"40px\")\n",
    ")\n",
    "\n",
    "run_btn = widgets.Button(\n",
    "    description=\"  여행 일정 생성하기  \",\n",
    "    button_style=\"primary\",\n",
    "    icon=\"paper-plane\",\n",
    "    layout=widgets.Layout(width=\"220px\", height=\"46px\")\n",
    ")\n",
    "\n",
    "output_area = widgets.Output()\n",
    "\n",
    "# ── 레이아웃\n",
    "display(widgets.VBox([\n",
    "    city_input,\n",
    "    widgets.HBox([start_date_input]),\n",
    "    widgets.HBox([days_input, days_label]),\n",
    "    widgets.HBox([dog_toggle, car_toggle], layout=widgets.Layout(gap=\"12px\", margin=\"8px 0\")),\n",
    "    run_btn,\n",
    "    output_area,\n",
    "], layout=widgets.Layout(gap=\"10px\")))\n",
    "\n",
    "# ── 버튼 클릭 핸들러\n",
    "def on_run(b):\n",
    "    output_area.clear_output()\n",
    "    with output_area:\n",
    "        city     = city_input.value.strip() or \"파주\"\n",
    "        start_dt = datetime.combine(start_date_input.value, datetime.min.time())\n",
    "        days     = days_input.value + 1   # 슬라이더값 = 박수, +1 = 일수\n",
    "        with_dog = dog_toggle.value\n",
    "        has_car  = car_toggle.value\n",
    "\n",
    "        display(HTML(f\"<div style='padding:10px;color:#666'>⏳ <b>{city}</b> 날씨 조회 중...</div>\"))\n",
    "\n",
    "        # 1. 날씨 조회\n",
    "        weather_list = get_midterm_weather(city, start_dt, days)\n",
    "\n",
    "        # 2. 날씨 카드 표시\n",
    "        render_weather_cards(weather_list, city)\n",
    "\n",
    "        cond_text = []\n",
    "        if with_dog: cond_text.append(\"🐾 애견동반\")\n",
    "        if has_car:  cond_text.append(\"🚗 자차이용\")\n",
    "        else:        cond_text.append(\"🚌 대중교통\")\n",
    "\n",
    "        display(HTML(f\"\"\"\n",
    "        <div style=\"background:#f0f4ff;border-radius:10px;padding:12px 16px;margin:12px 0;\n",
    "                    font-size:13px;color:#444\">\n",
    "          🤖 Claude AI가 <b>{city}</b> {days}일 일정 생성 중...\n",
    "          &nbsp; {'  '.join(cond_text)}\n",
    "        </div>\n",
    "        \"\"\"))\n",
    "\n",
    "        # 3. AI 일정 생성\n",
    "        plan = generate_travel_plan(city, weather_list, with_dog, has_car)\n",
    "\n",
    "        # 4. 결과 렌더링\n",
    "        render_plan_html(plan)\n",
    "\n",
    "        display(HTML(\"\"\"\n",
    "        <div style=\"text-align:center;padding:16px;color:#aaa;font-size:12px;margin-top:16px\">\n",
    "          날씨 출처: 기상청 중기예보 API &nbsp;|&nbsp; 일정: Claude AI\n",
    "        </div>\n",
    "        \"\"\"))\n",
    "\n",
    "run_btn.on_click(on_run)\n",
    "print(\"✅ UI 로드 완료 — 위의 폼을 채우고 버튼을 클릭하세요!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 📋 사용 방법\n",
    "\n",
    "| 단계 | 내용 |\n",
    "|------|------|\n",
    "| 1 | **CELL 1** 실행 → 패키지 설치 |\n",
    "| 2 | **CELL 2** 에서 API 키 두 개 입력 후 실행 |\n",
    "| 3 | **CELL 3~6** 순서대로 실행 |\n",
    "| 4 | **CELL 7** 실행 → 폼 입력 후 버튼 클릭 |\n",
    "\n",
    "## 🔑 API 키 발급 방법\n",
    "\n",
    "### 기상청 API\n",
    "1. [공공데이터포털](https://www.data.go.kr) 접속\n",
    "2. `기상청_중기예보` 검색\n",
    "3. **중기육상예보조회** + **중기기온조회** 두 개 활용 신청\n",
    "4. 승인 후 마이페이지 → 인증키 복사\n",
    "\n",
    "### Claude API\n",
    "1. [console.anthropic.com](https://console.anthropic.com) 접속\n",
    "2. API Keys → Create Key\n",
    "3. `sk-ant-...` 형태의 키 복사\n",
    "\n",
    "## ⚠️ 주의사항\n",
    "- 기상청 중기예보는 **오늘 기준 3~10일 후** 날씨만 제공됩니다\n",
    "- API 키 미설정 시 샘플 날씨 데이터로 동작합니다\n",
    "- 기상청 API 키는 신청 후 **1~2일 후** 활성화됩니다"
   ]
  }
 ]
}"


SyntaxError: unterminated string literal (detected at line 525) (3094145565.py, line 525)

In [46]:
# python
# 🔥 Hugging Face 설정
HF_API_KEY = "hf_xxxxxxxxxxxxx"

import requests

def generate_travel_plan(city, weather_list, with_dog, has_car):

    weather_summary = "\n".join([
        f"{w['day_label']} {w['weather']} {w['min_temp']}~{w['max_temp']}도"
        for w in weather_list
    ])

    prompt = f"""
    한국 여행 일정 만들어줘.

    도시: {city}
    조건: {'애견동반' if with_dog else '일반'} / {'자차' if has_car else '대중교통'}

    날씨:
    {weather_summary}

    일정은 날짜별로 오전/오후/저녁 나눠서 구체적으로 작성.
    """

    API_URL = "https://api-inference.huggingface.co/models/google/flan-t5-large"
    headers = {"Authorization": f"Bearer {HF_API_KEY}"}

    response = requests.post(
        API_URL,
        headers=headers,
        json={"inputs": prompt},
        timeout=30
    )

    try:
        result = response.json()
        return result[0]["generated_text"]
    except:
        return "❌ AI 생성 실패 (API 제한 또는 토큰 확인 필요)"
```


SyntaxError: invalid syntax (2084114403.py, line 41)

In [48]:
# python
# ============================================
# 1️⃣ 라이브러리 설치
# ============================================
!pip install requests ipywidgets -q

import requests
from datetime import datetime, timedelta
import ipywidgets as widgets
from IPython.display import display, clear_output

# ============================================
# 2️⃣ API KEY 입력 (본인 키로 변경)
# ============================================
KMA_API_KEY = "ab8817dc5f825af6b9573f9b8b87f41ed03db0fce3fc0ac6ce64064d65deb29a"
HF_API_KEY = "여기에_HF_API키"

# ============================================
# 3️⃣ 도시 코드
# ============================================
CITY_CODE = {
    "서울": "11B00000",
    "부산": "11H20000",
    "대구": "11H10000",
    "광주": "11F20000",
    "대전": "11C20000",
    "강릉": "11D20000",
    "제주": "11G00000",
}

# ============================================
# 4️⃣ 기상청 API 함수
# ============================================
def get_weather(city, days):
    if city not in CITY_CODE:
        city = "서울"

    now = datetime.now()
    tmFc = now.strftime("%Y%m%d") + ("0600" if now.hour < 18 else "1800")

    url = "https://apis.data.go.kr/1360000/MidFcstInfoService/getMidLandFcst"

    params = {
        "serviceKey": KMA_API_KEY,
        "pageNo": 1,
        "numOfRows": 10,
        "dataType": "JSON",
        "regId": CITY_CODE[city],
        "tmFc": tmFc,
    }

    res = requests.get(url, params=params)
    data = res.json()

    try:
        item = data["response"]["body"]["items"]["item"][0]
        result = []
        for i in range(3, 3 + days):
            weather = item.get(f"wf{i}Am", "정보없음")
            result.append(weather)
        return result
    except:
        return ["맑음", "구름", "비", "흐림"][:days]

# ============================================
# 5️⃣ AI 일정 생성 (무료)
# ============================================
def generate_plan(city, weather, dog, car):
    prompt = f"""
    한국 {city} 여행 일정 생성

    날씨: {weather}
    조건: {'애견동반' if dog else '일반'}, {'자차' if car else '대중교통'}

    일정은 날짜별로 오전/오후/저녁 나눠 작성
    """

    API_URL = "https://api-inference.huggingface.co/models/google/flan-t5-large"
    headers = {"Authorization": f"Bearer {HF_API_KEY}"}

    res = requests.post(API_URL, headers=headers, json={"inputs": prompt})

    try:
        return res.json()[0]["generated_text"]
    except:
        return "AI 생성 실패 (토큰 확인)"

# ============================================
# 6️⃣ UI 구성
# ============================================
city = widgets.Text(value="서울", description="도시:")
days = widgets.IntSlider(value=3, min=1, max=7, description="기간:")
dog = widgets.Checkbox(value=False, description="애견동반")
car = widgets.Checkbox(value=True, description="자차")
btn = widgets.Button(description="여행 생성")
output = widgets.Output()

def run(b):
    clear_output(wait=True)
    display(city, days, dog, car, btn, output)

    w = get_weather(city.value, days.value)
    plan = generate_plan(city.value, w, dog.value, car.value)

    with output:
        print("📍 도시:", city.value)
        print("🌤️ 날씨:", w)
        print("\n📅 여행 일정\n")
        print(plan)

btn.on_click(run)

display(city, days, dog, car, btn, output)
```


SyntaxError: invalid syntax (2772334532.py, line 114)

In [50]:
# ============================================
# 1️⃣ 라이브러리
# ============================================
!pip install requests ipywidgets -q

import requests
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output

# ============================================
# 2️⃣ 기상청 API 키 (필수)
# ============================================
KMA_API_KEY = "ab8817dc5f825af6b9573f9b8b87f41ed03db0fce3fc0ac6ce64064d65deb29a"

# ============================================
# 3️⃣ 도시 코드
# ============================================
CITY_CODE = {
    "서울": "11B00000",
    "부산": "11H20000",
    "대구": "11H10000",
    "광주": "11F20000",
    "대전": "11C20000",
    "강릉": "11D20000",
    "제주": "11G00000",
}

# ============================================
# 4️⃣ 날씨 가져오기
# ============================================
def get_weather(city, days):
    if city not in CITY_CODE:
        city = "서울"

    now = datetime.now()
    tmFc = now.strftime("%Y%m%d") + ("0600" if now.hour < 18 else "1800")

    url = "https://apis.data.go.kr/1360000/MidFcstInfoService/getMidLandFcst"

    params = {
        "serviceKey": KMA_API_KEY,
        "pageNo": 1,
        "numOfRows": 10,
        "dataType": "JSON",
        "regId": CITY_CODE[city],
        "tmFc": tmFc,
    }

    try:
        res = requests.get(url, params=params)
        item = res.json()["response"]["body"]["items"]["item"][0]

        weather = []
        for i in range(3, 3 + days):
            weather.append(item.get(f"wf{i}Am", "맑음"))

        return weather

    except:
        return ["맑음", "구름", "비", "흐림"][:days]

# ============================================
# 5️⃣ 🔥 AI 없이 자동 일정 생성 (핵심)
# ============================================
def generate_plan(city, weather, dog, car):

    indoor = ["박물관", "카페", "쇼핑몰", "전시관"]
    outdoor = ["공원", "관광지", "해변", "산책로"]

    plan = f"\n📍 {city} 여행 일정\n\n"

    for i, w in enumerate(weather):
        plan += f"📅 {i+1}일차 ({w})\n"

        if "비" in w:
            place = indoor
        else:
            place = outdoor

        plan += f"🌅 오전: {place[0]} 방문\n"
        plan += f"🌞 오후: {place[1]} 이동\n"
        plan += f"🌙 저녁: 맛집 탐방\n"

        if dog:
            plan += "🐾 애견 동반 가능 장소 추천\n"

        if car:
            plan += "🚗 드라이브 코스 포함\n"
        else:
            plan += "🚌 대중교통 이동\n"

        plan += "\n"

    return plan

# ============================================
# 6️⃣ UI
# ============================================
city = widgets.Text(value="서울", description="도시:")
days = widgets.IntSlider(value=3, min=1, max=7, description="기간:")
dog = widgets.Checkbox(value=False, description="애견동반")
car = widgets.Checkbox(value=True, description="자차")
btn = widgets.Button(description="여행 생성")
output = widgets.Output()

def run(b):
    clear_output(wait=True)
    display(city, days, dog, car, btn, output)

    w = get_weather(city.value, days.value)
    plan = generate_plan(city.value, w, dog.value, car.value)

    with output:
        print("🌤️ 날씨:", w)
        print(plan)

btn.on_click(run)

display(city, days, dog, car, btn, output)


Text(value='서울', description='도시:')

IntSlider(value=3, description='기간:', max=7, min=1)

Checkbox(value=True, description='애견동반')

Checkbox(value=True, description='자차')

Button(description='여행 생성', style=ButtonStyle())

Output()

## 8-4. 다중 그룹 기준

In [ ]:
# 부서 + 계급별 평균 급여
multi = df.groupby(['부서', '계급'])['급여'].mean().round(1)
print("[ 부서 × 계급별 평균 급여 ]")
print(multi)

## 8-5. 피벗 테이블 (Pivot Table) — 엑셀 스타일

In [ ]:
pivot = df.pivot_table(
    index='부서',       # 행
    columns='계급',      # 열
    values='훈련점수',   # 집계 대상
    aggfunc='mean'       # 집계 함수
).round(2)

print("[ 피벗 테이블: 부서 × 계급별 평균 훈련점수 ]")
print(pivot)

## 8-6. 그룹별 집계 결과 시각화

In [ ]:
# 부서별 평균 급여 막대그래프
dept_pay = df.groupby('부서')['급여'].mean().round(1)

plt.figure(figsize=(8, 4))
bars = plt.bar(dept_pay.index, dept_pay.values,
               color=['#2E86AB', '#5BA3C7', '#A7CFE0'], edgecolor='black')

for bar, v in zip(bars, dept_pay.values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'{v:.1f}', ha='center', fontweight='bold')

plt.title("부서별 평균 급여", fontsize=14, fontweight='bold')
plt.ylabel("평균 급여 (만원)")
plt.ylim(0, max(dept_pay.values) * 1.2)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
### 🔥 실습문제 8
`df` 데이터에 대해 다음을 수행하세요.

1. **계급별** 평균 급여, 평균 훈련점수, 인원수를 한 번에 출력
2. **부서별** 훈련점수의 **최대값**과 **최소값** 을 계산하여 표로 출력
3. **입사연도별** 평균 출석률 계산 (연도 오름차순 정렬)
4. **피벗 테이블**: 부서(행) × 계급(열)의 **평균 출석률** 생성

In [ ]:
# ✍️ 여기에 코드를 작성하세요



**✅ 정답 예시**

In [ ]:
# 1) 계급별 종합 집계
print("[ 1. 계급별 종합 통계 ]")
result1 = df.groupby('계급').agg(
    평균급여   = ('급여', 'mean'),
    평균점수   = ('훈련점수', 'mean'),
    인원수     = ('이름', 'count')
).round(2)
print(result1)

# 2) 부서별 훈련점수 max/min
print("\n[ 2. 부서별 훈련점수 범위 ]")
result2 = df.groupby('부서')['훈련점수'].agg(['max', 'min']).round(2)
print(result2)

# 3) 입사연도별 평균 출석률
print("\n[ 3. 입사연도별 평균 출석률 ]")
result3 = df.groupby('입사연도')['출석률'].mean().round(2).sort_index()
print(result3)

# 4) 피벗 테이블
print("\n[ 4. 부서 × 계급 피벗 - 평균 출석률 ]")
pivot = df.pivot_table(
    index='부서', columns='계급',
    values='출석률', aggfunc='mean'
).round(2)
print(pivot)

---
# 🏆 종합 실습 프로젝트 — 데이터 전처리 파이프라인

> 🎯 **목표**: 원본 데이터를 불러와 **결측값 → 중복 → 타입 변환 → 이상치 → 정규화 → 집계** 까지  
> 완결된 **전처리 파이프라인**을 구축합니다.

## 요구사항 체크리스트

- [ ] ① 원본 CSV 불러오기 + shape 확인
- [ ] ② 결측값 확인 및 처리 (급여·훈련점수·출석률)
- [ ] ③ 중복 행 제거 (이름 기준)
- [ ] ④ 타입 변환 (입대일 → datetime)
- [ ] ⑤ 이상치 제거 (훈련점수 IQR 방법)
- [ ] ⑥ 정규화 (급여 Min-Max)
- [ ] ⑦ 그룹 집계 (부서별 평균)
- [ ] ⑧ 시각화 (막대그래프 + 박스플롯)
- [ ] ⑨ 정제된 데이터 저장

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

print("=" * 50)
print("🚀 데이터 전처리 파이프라인 시작")
print("=" * 50)

# ────────────────────────────────
# ① 데이터 불러오기
# ────────────────────────────────
df = pd.read_csv('/content/soldiers.csv')
print(f"\n① 원본: {df.shape[0]}행 × {df.shape[1]}열")

# ────────────────────────────────
# ② 결측값 처리
# ────────────────────────────────
print(f"\n② 결측값 처리 전:")
print(df.isnull().sum()[df.isnull().sum() > 0])

df['급여']     = df['급여'].fillna(df['급여'].mean())
df['훈련점수'] = df['훈련점수'].fillna(df['훈련점수'].median())
df['출석률']   = df['출석률'].fillna(df['출석률'].median())

print(f"\n   ✅ 결측값 처리 후: {df.isnull().sum().sum()}개")

# ────────────────────────────────
# ③ 중복 제거
# ────────────────────────────────
before = len(df)
df = df.drop_duplicates(subset=['이름'], keep='first').reset_index(drop=True)
print(f"\n③ 중복 제거: {before} → {len(df)}행 ({before-len(df)}개 제거)")

# ────────────────────────────────
# ④ 타입 변환
# ────────────────────────────────
df['입대일'] = pd.to_datetime(df['입대일'])
df['입대연도'] = df['입대일'].dt.year
print(f"\n④ 타입 변환 완료 (입대일 → datetime)")

# ────────────────────────────────
# ⑤ 이상치 제거 (IQR)
# ────────────────────────────────
col = '훈련점수'
Q1, Q3 = df[col].quantile([0.25, 0.75])
IQR = Q3 - Q1
lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR

out_mask = (df[col] < lower) | (df[col] > upper)
print(f"\n⑤ 이상치 탐지: {out_mask.sum()}개")
df = df[~out_mask].reset_index(drop=True)
print(f"   제거 후: {len(df)}행")

# ────────────────────────────────
# ⑥ 정규화
# ────────────────────────────────
mm = MinMaxScaler()
df['급여_정규화'] = mm.fit_transform(df[['급여']]).round(3)
print(f"\n⑥ 급여 정규화 완료 [0, 1] 범위로 변환")

# ────────────────────────────────
# ⑦ 집계
# ────────────────────────────────
print(f"\n⑦ 부서별 요약 통계:")
summary = df.groupby('부서').agg(
    인원수    = ('이름', 'count'),
    평균급여  = ('급여', 'mean'),
    평균점수  = ('훈련점수', 'mean'),
    평균출석  = ('출석률', 'mean')
).round(2)
print(summary)

# ────────────────────────────────
# ⑨ 정제 데이터 저장
# ────────────────────────────────
df.to_csv('/content/soldiers_clean.csv', index=False, encoding='utf-8-sig')
print(f"\n⑨ 저장 완료: /content/soldiers_clean.csv")
print(f"   최종: {df.shape[0]}행 × {df.shape[1]}열")
print("=" * 50)

In [ ]:
# ⑧ 시각화
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 왼쪽: 부서별 평균 급여 막대그래프
dept_pay = df.groupby('부서')['급여'].mean().round(1)
bars = axes[0].bar(dept_pay.index, dept_pay.values,
                   color=['#2E86AB', '#5BA3C7', '#A7CFE0'], edgecolor='black')
for bar, v in zip(bars, dept_pay.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                 f'{v:.1f}', ha='center', fontweight='bold')
axes[0].set_title("부서별 평균 급여", fontsize=13, fontweight='bold')
axes[0].set_ylabel("급여 (만원)")
axes[0].grid(axis='y', alpha=0.3)

# 오른쪽: 부서별 훈련점수 박스플롯
dept_list = df['부서'].unique()
data_list = [df[df['부서']==d]['훈련점수'].values for d in dept_list]
axes[1].boxplot(data_list, labels=dept_list)
axes[1].set_title("부서별 훈련점수 분포", fontsize=13, fontweight='bold')
axes[1].set_ylabel("훈련점수")
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✅ 전처리 파이프라인 완료! 다음 단계는 3일차 EDA·시각화입니다.")

---
### 🔥 최종 도전 과제
위 파이프라인을 확장하여 다음을 추가해 보세요.

1. **`"복무기간(일수)"`** 컬럼 추가 → 오늘 날짜 - 입대일
2. **계급 코드** 컬럼 추가 (이병=1, 일병=2, 상병=3, 병장=4)
3. **원-핫 인코딩** 적용 → `pd.get_dummies(columns=['부서'])`
4. **종합 점수** 컬럼 추가 → (훈련점수 × 0.6 + 출석률 × 0.4)
5. **종합 점수 상위 5명** 출력

💡 어려우면 ChatGPT/Claude에 Vibe Coding으로 요청하세요!

In [ ]:
# ✍️ 여기에 도전 과제 코드를 작성하세요



**✅ 도전 과제 정답 예시**

In [ ]:
from datetime import datetime

# 원본 정제된 데이터 다시 불러오기
df = pd.read_csv('/content/soldiers_clean.csv')
df['입대일'] = pd.to_datetime(df['입대일'])

# 1) 복무기간(일수)
today = pd.Timestamp.today().normalize()
df['복무기간일'] = (today - df['입대일']).dt.days

# 2) 계급 코드
rank_map = {'이병': 1, '일병': 2, '상병': 3, '병장': 4}
df['계급코드'] = df['계급'].map(rank_map)

# 3) 원-핫 인코딩 (부서)
df_encoded = pd.get_dummies(df, columns=['부서'], prefix='부서')

# 4) 종합 점수
df['종합점수'] = (df['훈련점수'] * 0.6 + df['출석률'] * 0.4).round(2)

# 5) 상위 5명
top5 = df.sort_values('종합점수', ascending=False).head(5)
print("🏆 종합점수 TOP 5")
print(top5[['이름', '계급', '부서', '훈련점수', '출석률', '종합점수']].to_string(index=False))

print("\n[ 원-핫 인코딩 결과 (앞 5행) ]")
print(df_encoded.head().to_string())

---
# 🎓 2일차 학습 정리

## ✅ 오늘 배운 내용

| 챕터 | 핵심 내용 | 주요 함수 |
|---|---|---|
| **1. 데이터 수집** | 공공데이터·API·파일 | `requests.get()` |
| **2. 파일 읽기** | CSV·Excel·JSON | `pd.read_csv/excel/json()` |
| **3. 데이터 탐색** | 구조·통계 파악 | `head/info/describe/shape` |
| **4. 선택·필터링** | 라벨/위치/조건 | `loc/iloc`, Boolean Indexing |
| **5. 결측값 처리** | 삭제·채우기 | `isnull/dropna/fillna` |
| **6. 타입·정규화** | 변환·스케일링 | `astype/MinMaxScaler` |
| **7. 이상치 처리** | IQR·제거·대체 | `quantile/clip` |
| **8. 그룹화·집계** | split-apply-combine | `groupby/agg/pivot_table` |

## 🔄 전처리 표준 파이프라인
```
수집 → 탐색 → 결측값 처리 → 중복 제거 → 타입 변환 →
이상치 제거 → 정규화 → 분석 준비 완료 🚀
```

## 📅 다음 일정
- **3일차**: 데이터 분석 및 가공 (그룹화 심화·피벗·상관분석·시각화)
- **4일차**: 미니 프로젝트 — 부대 실무 데이터 EDA

---
**스마트 강군 육성, 국방 AX(AI 전환)에 만전을 기하겠습니다** 🎖️

*교육기관: (사)한국오픈소스협회 | 문의: 02-6012-7414 / kmil@osskorea.org*